# Testing Sparse Auto Encoder (SAE)

In [3]:
# imports
import tensorflow as tf
import keras
from keras import layers
import numpy as np
import Bio
import transformers
from objects.autoencoder import SparseAutoEncoder

In [ ]:
# create dataset from random tensors to test
SAE_name = 'autoencoder_test'
embed_length = 2048
ef = 4

print("=== Generating Test Data ===")
fake_embeddings = tf.random.uniform(shape=[1000, embed_length])
fake_dataset = tf.data.Dataset.from_tensor_slices((fake_embeddings, fake_embeddings)).batch(100)

print("=== Initializing Model ===")
# initialize the optimizer
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

# initialize the loss function
loss = keras.losses.MeanSquaredError()

# initialize the metrics
metrics = [
    keras.metrics.MeanSquaredError(),
    keras.metrics.KLDivergence(),
]

# compile the model
autoencoder = SparseAutoEncoder(encoding_size=embed_length, expansion_factor=ef)
autoencoder.compile(optimizer=optimizer, loss=loss, metrics=metrics)

print(autoencoder.summary())

tb_callback = keras.callbacks.TensorBoard(log_dir=f'./logs/{SAE_name}', histogram_freq=5)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='mean_squared_error',
    min_delta=0.001,
    patience=10, 
    restore_best_weights=True
    )

print("=== Training Model ===")
history = autoencoder.fit(fake_dataset, epochs=100, callbacks=[tb_callback, early_stopping])
print("=== Saving Model ===")
path = f'./models/{SAE_name}.keras'
#autoencoder.save(f'./models/{SAE_name}.keras')
keras.models.save_model(autoencoder, path)
print(f"Model saved to: {path}")
print(autoencoder.summary())

=== Generating Test Data ===
=== Initializing Model ===


TypeError: kl_divergence() missing 2 required positional arguments: 'y_true' and 'y_pred'

In [ ]:
feat_weights = np.array(autoencoder.weights[2])
print(f"maximum: {max(feat_weights)}")
print(f"minimum: {min(feat_weights)}")

In [ ]:
autoencoder.predict_on_batch(fake_embeddings[:100])

In [ ]:
autoencoder.last_features

In [ ]:
dir(autoencoder.last_features)

In [ ]:
check_sparsity = autoencoder.last_features
check_sparsity.numpy

In [ ]:
keras.utils.is_keras_tensor(autoencoder.last_features)

In [ ]:
type(autoencoder.last_features)